# Rollout and Change Management

> **The Riverside situation:** Riverside is introducing an editorial assistant whose quality, authorization, adoption, and external actions can fail independently. The rollout must expose one bounded cohort, observe agreed signals, and stop before a local failure becomes broad customer impact.
>
> **Where you are:** Discovery bounded the workflow; architecture separated model, retrieval, policy, human, and tool authority; data and identity work defined fail-closed controls; service planning supplied provisional quality, latency, availability, cost, and support gates. You still cannot expose the candidate safely because a release report does not choose a customer cohort, resolve shadow disagreements, prepare users, or repair an action already committed.

> **What you finished last time:** service, capacity, and commercial assumptions tied to explicit evidence classes.
> **What this notebook delivers:** `ROL-01` rollout plan, `ROL-02` cohort/disagreement report, `ROL-03` decision record, `ROL-04` rollback/compensation drill, and `CHG-*` communications.
> **Prerequisite for the next notebook:** retained abort, containment, communication, and ownership evidence for incident response.

This notebook executed successfully against committed synthetic fixtures and produced the documented `HOLD` and `ABORT` decisions. Outputs were then cleared. The run verifies deterministic local calculations and fail-closed routing, not a customer rollout, cloud behavior, production readiness, or approval.

## 0 - The Challenge

> **The mission:** Riverside House - introduce an authorized editorial assistant without uncontrolled customer exposure or unowned committed actions.

**What we know so far:**

- Historical policy search takes 18 minutes at the median; bounded continuation drafting takes 42 minutes.
- Current guidance ranks first only 61% of the time, and editorial rework is 22%.
- The frozen route defines six cohorts from 80 offline cases to 72 employees.
- Forbidden access, prohibited autonomous action, duplicate workflow commits, and SEV-1 incidents are global aborts.
- **But we still cannot turn a candidate release into bounded, reversible customer change.**

**What's blocking us:** A launch date says when, not why. The first 200-task shadow window has a positive aggregate candidate delta, yet five policy blocks and only 80% disagreement-review coverage. Later, a canary improves cycle time but records one duplicate PageTurn commit. Without explicit gates, both failures can hide behind a healthy average.

**What this chapter unlocks:** A staged decision chain that holds incomplete shadow evidence, advances a qualified 12-editor champion cohort, aborts on a duplicate commit, routes new work to a known-good release, and separately reconciles the committed PageTurn action.

```mermaid
flowchart LR
    A["Calendar launch plan"] --> B["Failure: aggregate hides unsafe slices"]
    B --> C["Shadow disagreement review"]
    C --> D["Canary observation windows"]
    D --> E["Abort: duplicate commit"]
    E --> F["Deployment rollback"]
    F --> G["Committed-action reconciliation"]
    G --> H["Owned communication"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Load frozen and chapter-owned fixtures ----------------------------------
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path


def find_chapter_dir(start: Path) -> Path:
    for root in (start, *start.parents):
        direct = root / "fixtures" / "rollout-observations-v1.json"
        nested = root / "learning" / "role-based-tracks" / "fde" / "06-rollout-and-change-management" / "fixtures" / "rollout-observations-v1.json"
        if direct.exists():
            return root
        if nested.exists():
            return nested.parent.parent
    raise FileNotFoundError("Run from the repository or chapter directory.")


CHAPTER_DIR = find_chapter_dir(Path.cwd().resolve())
SHARED_DIR = CHAPTER_DIR.parent / "shared" / "fixtures"
with (CHAPTER_DIR / "fixtures" / "rollout-observations-v1.json").open(encoding="utf-8") as file:
    observations = json.load(file)
with (SHARED_DIR / "riverside-engagement-v1.json").open(encoding="utf-8") as file:
    engagement = json.load(file)

print(f"Fixture: {observations['fixture_version']}")
print(f"Shared contract: {engagement['fixture_version']}")
print("Local calculations remain synthetic fixture evidence.")

## 1 - Baseline Before Exposure

Historical decisions are a baseline, not perfect labels: people can use superseded policy, skip evidence, or correct an error later. Preserve the old workflow's time, rework, evidence freshness, workflow-error, and volume measures separately.

```mermaid
flowchart LR
    A["Historical workflow"] --> B["Sample tasks and slices"]
    B --> C["Decision plus source version"]
    C --> D{"Golden truth?"}
    D -->|Assume yes| E["Failure: old errors become policy"]
    D -->|Review| F["Disagreement categories"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Compare only with the candidate target | You cannot tell whether the workflow improved |
| Wrong | Treat every historical decision as correct | Old errors become a false golden set |
| Right | Preserve population, window, slices, and limitations | The comparison remains reviewable |
| Right | Adjudicate with current evidence | Baseline corrections stay explicit |

**Quick Health Check:** fixture versions, required IDs, evidence class, values, and units must match the frozen case.

In [ ]:
# -- Inspect and verify the historical baseline ------------------------------
baseline = {item["metric_id"]: item for item in observations["historical_baseline"]["metrics"]}
shared_metrics = {item["metric_id"]: item for item in engagement["baseline_metrics"]}
expected_ids = {"MET-RIV-001", "MET-RIV-003", "MET-RIV-004", "MET-RIV-005", "MET-RIV-006"}
assert set(baseline) == expected_ids
assert observations["shared_fixture_version"] == engagement["fixture_version"]
assert observations["historical_baseline"]["evidence_class"] == "measured_baseline"
for metric_id in sorted(baseline):
    metric = baseline[metric_id]
    assert metric["value"] == shared_metrics[metric_id]["value"]
    assert metric["unit"] == shared_metrics[metric_id]["unit"]
    print(f"{metric_id}: {metric['name']} = {metric['value']} {metric['unit']}")
print("PASS: chapter baseline matches frozen Riverside measurements.")

## 2 - Shadow Disagreement Review

Shadow mode replays work without serving candidate output. Mutating tools stay disabled or use a sandbox. Compare candidate and historical decisions within the same task, tenant, policy, and evidence slice.

Review coverage is the share of candidate-versus-baseline disagreements that received a disposition. It shows whether Riverside reviewed every disagreement; it does not show that the candidate was correct.

```mermaid
flowchart LR
    A["200 historical tasks"] --> B["Baseline decision"]
    A --> C["Candidate shadow decision"]
    B --> D["Slice-aligned comparison"]
    C --> D
    D --> E["Agreement / candidate / historical"]
    D --> F["Policy block or unresolved"]
    F --> G["HOLD until dispositioned"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** `SHADOW-RIV-001` has a positive candidate delta. Does the gate (A) advance, (B) hold because policy blocks and review coverage fail, or (C) roll back a live deployment?

In [ ]:
# -- Calculate shadow evidence and close the prediction loop ----------------
def summarize_shadow(window):
    total = sum(row["tasks"] for row in window["slices"])
    agreement = sum(row["agreement"] for row in window["slices"])
    candidate_preferred = sum(row["candidate_preferred"] for row in window["slices"])
    historical_preferred = sum(row["historical_preferred"] for row in window["slices"])
    policy_blocks = sum(row["policy_block"] for row in window["slices"])
    unresolved = sum(row["unresolved"] for row in window["slices"])
    disagreements = total - agreement
    reviewed = sum(row["reviewed_disagreements"] for row in window["slices"])
    coverage = reviewed / disagreements if disagreements else 1.0
    delta_pp = 100 * (candidate_preferred - historical_preferred) / total
    gate = window["required_gate"]
    checks = {
        "sample_size": total >= gate["minimum_tasks"],
        "review_coverage": coverage >= gate["minimum_disagreement_review_coverage"],
        "policy_blocks": policy_blocks <= gate["maximum_policy_blocks"],
        "forbidden_access": window["forbidden_access_count"] <= gate["maximum_forbidden_accesses"],
        "candidate_delta": delta_pp >= gate["minimum_candidate_preference_delta_percentage_points"],
    }
    return {"total": total, "disagreements": disagreements, "reviewed": reviewed, "coverage": coverage, "policy_blocks": policy_blocks, "unresolved": unresolved, "delta_pp": delta_pp, "checks": checks, "decision": "GO" if all(checks.values()) else "HOLD"}

first_shadow = summarize_shadow(observations["shadow_windows"][0])
print(f"Candidate delta: {first_shadow['delta_pp']:.1f} percentage points")
print(f"Review coverage: {first_shadow['coverage']:.1%}; policy blocks: {first_shadow['policy_blocks']}")
print(f"Decision: {first_shadow['decision']}")
assert first_shadow["decision"] == "HOLD"
print("Prediction B confirmed: a healthy aggregate cannot override failed critical gates.")

In [ ]:
# -- Re-evaluate after remediation and check fixture consistency ------------
second_shadow = summarize_shadow(observations["shadow_windows"][1])
for check, passed in second_shadow["checks"].items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")
assert second_shadow["decision"] == "GO"
for window in observations["shadow_windows"]:
    for row in window["slices"]:
        category_total = sum(row[name] for name in ["agreement", "candidate_preferred", "historical_preferred", "policy_block", "unresolved"])
        assert category_total == row["tasks"]
assert second_shadow["total"] == 200
assert second_shadow["reviewed"] == second_shadow["disagreements"]
assert observations["shadow_windows"][1]["forbidden_access_count"] == 0
print("GO: corrected evidence authorizes the champion canary only.")

**Your turn:** Change the proposed review threshold below. The check rejects a policy that weakens the frozen 100% gate.

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Report only aggregate agreement | Critical tenant, policy, or action slices disappear |
| Wrong | Let shadow calls write to PageTurn | Shadow becomes unapproved production exposure |
| Right | Predeclare categories and review every disagreement | The gate cannot be tuned after the result |
| Right | Preserve release, data, evaluator, policy, and slice versions | Comparisons remain valid |

**Quick Health Check:** category totals match tasks; the passing window has 200 tasks, complete review, and zero forbidden access.

In [ ]:
# -- Drill: do not tune the gate after seeing the result --------------------
proposed_review_coverage = 1.00  # CHANGE THIS: try 0.80
canonical_review_coverage = observations["shadow_windows"][0]["required_gate"]["minimum_disagreement_review_coverage"]
if proposed_review_coverage < canonical_review_coverage:
    print("FAIL: the proposal weakens a predeclared gate after seeing the result.")
else:
    print("PASS: the proposal preserves or strengthens the approved gate.")

## 3 - Canary Cohorts Are Change Boundaries

A canary is not just 10% traffic. For Riverside it is a named group with a business owner, technical owner, approved workflows, training, support, manual fallback, structured feedback, and a bounded exposure mode.

```mermaid
flowchart LR
    A["Offline: 80 cases"] --> B["Shadow: 200 tasks"]
    B --> C["Champions: 12 users\nread and draft"]
    C --> D["Alder: 28 users\nconfirmed proposal"]
    C --> E["Northbank: 18 users\nUS route"]
    D --> F["Broad: 72 employees"]
    E --> F
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Nine of 12 champions were active, 10 supplied structured feedback, no forbidden access occurred, and the window lasted seven days. Does the gate pass, hold for adoption, or abort for safety?

Customer change management belongs inside the gate: explain allowed behavior, train the workflow, preserve the manual path, test feedback, publish support hours, and state that participation is not broad approval.

**Common Pitfalls:** random users plus a traffic percentage lose workflow ownership; logins are not useful adoption. Name users, use cases, training, fallback, support, and structured rejection reasons.

In [ ]:
# -- Inspect cohort ownership and evaluate the champion gate ----------------
cohorts = engagement["rollout"]["cohorts"]
for cohort in cohorts:
    print(f"{cohort['cohort_id']}: {cohort['size']} {cohort['unit']} | {cohort['mode']} | business={cohort['business_owner_person_id']} | technical={cohort['technical_owner_person_id']}")
assert [cohort["cohort_id"] for cohort in cohorts] == [f"COH-RIV-00{index}" for index in range(6)]

champion = observations["canary_windows"][0]
gate = champion["required_gate"]
active_rate = champion["weekly_active_users"] / champion["eligible_users"]
feedback_coverage = champion["users_with_structured_feedback"] / champion["eligible_users"]
checks = {
    "active": active_rate >= gate["minimum_weekly_active_rate"],
    "feedback": feedback_coverage > gate["minimum_structured_feedback_coverage"],
    "safety": champion["forbidden_access_count"] <= gate["maximum_forbidden_accesses"],
    "time": champion["observation_days"] >= gate["minimum_observation_days"],
}
champion_decision = "GO" if all(checks.values()) else "HOLD"
shared_champion = next(item for item in cohorts if item["cohort_id"] == "COH-RIV-002")
assert champion["eligible_users"] == shared_champion["size"] == 12
assert shared_champion["mode"] == "read_and_draft_only"
assert engagement["support_and_handoff"]["covered_hours"]
print(f"Active: {active_rate:.1%}; feedback: {feedback_coverage:.1%}; decision: {champion_decision}")
assert champion_decision == "GO"
print("Prediction confirmed: GO authorizes Alder only, not broad availability.")

## 4 - Observation Windows and Ramp Gates

An observation window is part of the claim. Seven quiet days cannot satisfy a fourteen-day gate, and windows cannot be merged if releases, cohorts, policy, or measurement definitions changed. Compare the candidate cycle time with Riverside's historical cycle time and report the percentage reduction; that measures change, not causality or acceptable quality.

```mermaid
flowchart LR
    A["Alder week 1\ncycle time improves"] --> B{"14-day gate complete?"}
    B -->|No| C["HOLD current exposure"]
    C --> D["Alder week 2"]
    D --> E{"All gates pass?"}
    E -->|Yes| F["Consider next cohort"]
    E -->|Duplicate commit| G["ABORT immediately"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Common Pitfalls:** ending a window when the result looks good creates optional stopping; merging changed releases creates incomparable evidence. Predeclare duration, population, metrics, slices, and freshness, and give safety aborts precedence.

**Quick Health Check:** stable release/cohort IDs, complete duration, fresh signals, independent gates, and global abort visibility.

In [ ]:
# -- Evaluate observation windows with abort precedence ---------------------
alder_windows = observations["canary_windows"][1:]
alder_gate = observations["alder_required_gate"]


def evaluate_alder(windows):
    days = sum(window["observation_days"] for window in windows)
    baseline_minutes = windows[0]["baseline_policy_cycle_minutes"]
    candidate_minutes = sum(window["candidate_policy_cycle_minutes"] for window in windows) / len(windows)
    improvement = (baseline_minutes - candidate_minutes) / baseline_minutes
    forbidden = sum(window["forbidden_access_count"] for window in windows)
    duplicates = sum(window["duplicate_workflow_commit_count"] for window in windows)
    quality = sum(window["quality_regression_count"] for window in windows)
    max_cost = max(window["cost_per_successful_request_usd"] for window in windows)
    max_support = max(window["support_ticket_count"] for window in windows)
    abort = forbidden > alder_gate["maximum_forbidden_accesses"] or duplicates > alder_gate["maximum_duplicate_workflow_commits"]
    pass_other = improvement >= alder_gate["minimum_cycle_time_improvement_rate"] and quality <= alder_gate["maximum_quality_regressions"] and max_cost <= alder_gate["maximum_cost_per_successful_request_usd"] and max_support <= alder_gate["maximum_support_tickets_per_week"]
    decision = "ABORT" if abort else ("GO" if days >= alder_gate["minimum_observation_days"] and pass_other else "HOLD")
    return {"days": days, "improvement": improvement, "duplicates": duplicates, "forbidden": forbidden, "decision": decision}


week_one = evaluate_alder(alder_windows[:1])
full_window = evaluate_alder(alder_windows)
print(f"Week 1: {week_one['improvement']:.1%}, {week_one['days']} days -> {week_one['decision']}")
print(f"Full window: {full_window['improvement']:.1%}, duplicate commits={full_window['duplicates']} -> {full_window['decision']}")
assert week_one["decision"] == "HOLD" and full_window["decision"] == "ABORT"
assert len({window["candidate_release_id"] for window in alder_windows}) == 1
assert len({window["cohort_id"] for window in alder_windows}) == 1
print("PASS: time blocks early promotion; global abort overrides outcome improvement.")

**Your turn:** Change the proposed observation window to seven days. The check below explains why shortening the gate after week one is invalid.

In [ ]:
# -- Drill: preserve the predeclared observation window ---------------------
proposed_observation_days = 14  # CHANGE THIS: try 7
canonical_days = alder_gate["minimum_observation_days"]
if proposed_observation_days < canonical_days:
    print("FAIL: shortening after a good week misses the later abort.")
else:
    print("PASS: the proposal preserves the approved observation requirement.")
assert sum(window["observation_days"] for window in alder_windows) == 14
assert max(window["cost_per_successful_request_usd"] for window in alder_windows) <= 0.05
assert full_window["duplicates"] == 1

## 5 - Abort, Deployment Rollback, and Committed Actions

The duplicate commit triggers two parallel tracks. Deployment rollback stops new candidate work: freeze the ramp, disable writes, and route traffic to `rel-riv-002`. Committed-action recovery asks what PageTurn already changed, queries by the same business idempotency key, preserves history, and applies a correction only with workflow-owner approval.

```mermaid
flowchart TD
    A["Duplicate commit detected"] --> B["Freeze ramp and preserve evidence"]
    B --> C["Track A: route new work to rel-riv-002"]
    B --> D["Track B: query PageTurn by business key"]
    C --> E["Candidate writes disabled"]
    D --> F{"Commit state?"}
    F -->|Confirmed| G["Owner-approved correction"]
    F -->|Absent| H["Approved retry with same key"]
    F -->|Unknown| I["Escalate; do not guess"]
    style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Is routing traffic to `rel-riv-002` enough to close the event? (A) Yes, (B) no because the PageTurn commit remains, or (C) retry with a new key.

**Common Pitfalls:** traffic zero does not mean external records are restored; a new retry key creates a new action; compensation before checking commit state is a guess.

**Quick Health Check:** known-good release, zero candidate writes, preserved business key/history, queried state, and separate business/technical owners.

In [ ]:
# -- Separate deployment recovery from committed-action recovery ------------
action = observations["committed_actions"][0]
release_pair = observations["release_pair"]
assert action["commit_status"] == "confirmed_committed"
assert release_pair["rollback_target_release_id"] == "rel-riv-002"
print("Track A - deployment rollback:")
print(f"  {action['deployment_response']}")
print("Track B - committed action:")
print(f"  {action['committed_action_response']}")
print(f"  business key: {action['business_idempotency_key']}")
print(f"  owners: {action['compensation_owner_person_id']} / {action['business_owner_person_id']}")
required = {"business_idempotency_key", "commit_status", "deployment_response", "committed_action_response", "compensation_owner_person_id", "business_owner_person_id"}
assert required.issubset(action)
assert action["before_state"] != action["after_state"]
assert "reconcile" in action["committed_action_response"]
print("Prediction B confirmed: traffic rollback cannot undo a committed external action.")

## 6 - Communications Are an Operational Control

Communication constrains customer behavior during uncertainty: which workflows remain safe, which actions are paused, where the manual fallback lives, who is investigating, and when the next evidence-backed update arrives.

```mermaid
flowchart LR
    A["Detection at 10:00 UTC"] --> B["Known facts"]
    A --> C["Unknowns with owners"]
    A --> D["Containment and fallback"]
    B --> E["SEV-2 acknowledgement by 11:00"]
    C --> F["Customer update by 12:00"]
    D --> F
    F --> G["Next decision time, not guessed ETA"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

A cohort invitation sets expectations before exposure. A status update preserves known and unknown facts. An abort notice names containment and committed-action status. Re-enablement names regression evidence, residual risk, approval, and temporary-control expiry.

**Common Pitfalls:** do not promise an ETA before state is known, say `rolled back` without committed-action status, or place customer content and IDs in broad channels.

**Quick Health Check:** UTC timestamps, release/cohort scope, known/unknown facts, fallback, committed-action status, owner, and next update.

In [ ]:
# -- Calculate and verify communication deadlines ---------------------------
clock = observations["communication_clock"]
detected = datetime.fromisoformat(clock["detected_at_utc"].replace("Z", "+00:00"))
ack_due = detected + timedelta(minutes=clock["acknowledgement_target_minutes"])
update_due = detected + timedelta(minutes=clock["customer_update_target_minutes"])
severity = next(item for item in engagement["support_and_handoff"]["severity_definitions"] if item["severity"] == clock["severity"])
assert clock["acknowledgement_target_minutes"] == severity["acknowledgement_target_minutes"]
assert clock["customer_update_target_minutes"] == severity["customer_update_minutes"]
assert clock["facts_known_at_detection"] and clock["facts_unknown_at_detection"]
print(f"Detected: {detected.astimezone(timezone.utc).isoformat()}")
print(f"Acknowledgement due: {ack_due.astimezone(timezone.utc).isoformat()}")
print(f"Customer update due: {update_due.astimezone(timezone.utc).isoformat()}")
print("PASS: deadlines come from severity policy, not improvisation.")

## 7 - Ownership and the Go / No-Go Record

A metric does not approve exposure. The business owner accepts workflow impact; the technical owner controls release; Security owns isolation; Operations owns rollback; the incident commander controls response; and the launch authority accepts bounded customer risk. The FDE supplies evidence and a recommendation, not unilateral approval.

```mermaid
flowchart LR
    A["Evidence package"] --> B["FDE recommendation"]
    B --> C["Workflow owner"]
    B --> D["Security and operations"]
    C --> E{"Named decision authority"}
    D --> E
    E -->|GO| F["Next named cohort only"]
    E -->|HOLD| G["Keep current exposure"]
    E -->|ABORT| H["Contain, roll back, reconcile"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Decision precedence: a global abort yields `ABORT`; incomplete or stale evidence yields `HOLD`; all passing gates may yield `GO` for one next stage. Conditional approval needs a condition, owner, due date, exposure limit, and automatic response.

**Quick Health Check:** release/cohort IDs, authority, business and technical owners, gate evidence, rollback target, conditions, expiry, communication, and retained dissent.

In [ ]:
# -- Build and verify bounded decision records -----------------------------
champion_record = {"decision": champion_decision, "current_cohort": "COH-RIV-002", "next_cohort": "COH-RIV-003", "candidate_release_id": champion["candidate_release_id"], "rollback_target": observations["release_pair"]["rollback_target_release_id"], "business_owner": "PER-RIV-002", "technical_owner": "PER-RIV-005"}
alder_record = {"decision": full_window["decision"], "current_cohort": "COH-RIV-003", "next_cohort": None, "candidate_release_id": alder_windows[-1]["candidate_release_id"], "rollback_target": observations["release_pair"]["rollback_target_release_id"], "incident_id": alder_windows[-1]["triggered_incident_id"], "business_owner": "PER-RIV-002", "technical_owner": "PER-RIV-005"}
required_fields = {"decision", "current_cohort", "candidate_release_id", "rollback_target", "business_owner", "technical_owner"}
for record in [champion_record, alder_record]:
    assert required_fields.issubset(record)
    assert record["decision"] in {"GO", "HOLD", "ABORT", "ROLLBACK"}
    assert record["rollback_target"] != record["candidate_release_id"]
assert champion_record["next_cohort"] == "COH-RIV-003"
assert alder_record["next_cohort"] is None and alder_record["decision"] == "ABORT"
print(f"Champion: {champion_record['decision']} -> {champion_record['next_cohort']}")
print(f"Alder: {alder_record['decision']} -> incident {alder_record['incident_id']}")
print("PASS: decisions are scoped, owned, reversible, and non-transitive.")

## 8 - Close the Loop

Evidence can hold a stage, advance one cohort, abort the next, reopen an earlier gate, and hand an incident package to operations. Exposure changes when evidence changes.

```mermaid
flowchart LR
    A["Historical baseline"] --> B["Shadow 1: HOLD"]
    B --> C["Shadow 2: GO"]
    C --> D["Champions: GO"]
    D --> E["Alder week 1: HOLD"]
    E --> F["Alder week 2: ABORT"]
    F --> G["Known-good release restored"]
    F --> H["Committed action reconciled"]
    G --> I["Incident and revalidation"]
    H --> I
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Riverside rollout checkpoint

- **Shadow:** hold until every disagreement is reviewed and policy blocks are cleared, even when the average result improves.
- **Champions:** advance only the named 12-editor cohort after adoption, feedback, safety, support, and time checks pass.
- **Alder:** keep exposure unchanged until the full fourteen-day window completes; abort immediately on a duplicate PageTurn commit.
- **Recovery:** route new work to the known-good release, then query and reconcile the PageTurn action as a separate business decision.
- **Authority:** a `GO` decision permits one next stage only and expires when the release, cohort, policy, evidence, or owner changes.

### Evidence boundary

The notebook checks Riverside's gate logic against synthetic fixtures. It does not prove production traffic shifting, live shadow mirroring, user readiness, PageTurn recovery, customer notification, or approval. Those require authorized environments, named owners, and retained drill or production evidence.

> **Forward:** carry the abort record, preserved evidence, committed-action state, customer update clock, and re-enablement owners into `07-incident-response-and-recovery`.